# ADK vs LangGraph — Framework Comparison

**Purpose:** Quick-reference notebook for Q&A during the ADK presentation.  
Each section shows side-by-side code snippets so you can point to concrete differences.

---

## 1. Agent & Task Management

**Q: How do you define an agent in each framework?**

ADK uses high-level, declarative classes. LangGraph uses low-level nodes and edges.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Define an agent in ~10 lines
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from google.adk.agents import LlmAgent

agent = LlmAgent(
    name="TicketAssistant",
    model="gemini-2.5-flash",
    instruction="You are a JIRA ticket assistant. Help users create tickets.",
    description="Creates JIRA tickets from user descriptions.",
    tools=[save_input, create_ticket],  # plain Python functions
)

# That's it. The agent handles conversation, tool calling, and response 
# generation automatically via the ReAct loop.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — Define the same agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
tools = [save_input, create_ticket]
llm_with_tools = llm.bind_tools(tools)

# Step 1: Define each node as a function
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_tools(state: MessagesState):
    # manually route and execute tool calls
    ...

# Step 2: Build the graph
graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tools)

# Step 3: Wire the edges (including conditional routing)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_call_tool, {"yes": "tools", "no": END})
graph.add_edge("tools", "agent")

app = graph.compile()

# More code, but you control every edge explicitly.

**Takeaway:** ADK abstracts the ReAct loop. LangGraph makes you build it yourself — more control, more code.

---
## 2. Workflow Management

**Q: How do you build multi-step pipelines?**

ADK gives you opinionated primitives. LangGraph uses conditional edges and cycles.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Sequential pipeline
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from google.adk.agents import SequentialAgent

pipeline = SequentialAgent(
    name="ETLPipeline",
    sub_agents=[extractor, transformer, validator],
)
# Runs extractor → transformer → validator, in order. Done.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Loop (draft → review → repeat)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from google.adk.agents import LoopAgent

review_loop = LoopAgent(
    name="DraftReviewLoop",
    sub_agents=[drafter, reviewer],  # reviewer calls exit_loop when satisfied
    max_iterations=3,
)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Parallel execution
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from google.adk.agents import ParallelAgent

parallel = ParallelAgent(
    name="DataGatherer",
    sub_agents=[fetch_db, fetch_api, fetch_cache],
)
# All three run concurrently. Results merged automatically.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — Same loop requires explicit graph construction
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langgraph.graph import StateGraph

graph = StateGraph(MyState)
graph.add_node("drafter", draft_node)
graph.add_node("reviewer", review_node)

graph.add_edge(START, "drafter")
graph.add_edge("drafter", "reviewer")

# The loop: reviewer decides whether to loop back or exit
graph.add_conditional_edges(
    "reviewer",
    lambda state: "drafter" if state["needs_revision"] else END
)

app = graph.compile()

# More flexible (any node can go anywhere), but you wire everything manually.

**Takeaway:** ADK has `SequentialAgent`, `LoopAgent`, `ParallelAgent` as building blocks. LangGraph is a blank canvas — powerful but verbose.

---
## 3. State Management

**Q: How does each framework handle conversation state?**

This is where LangGraph genuinely shines with checkpointing.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Session-based state with scoping
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Inside a tool:
def my_tool(query: str, tool_context: ToolContext) -> dict:
    # Session-scoped (this conversation only)
    tool_context.state["raw_input"] = query
    
    # User-scoped (persists across conversations for this user)
    tool_context.state["user:preferences"] = {"lang": "en"}
    
    # App-scoped (global, all users)
    tool_context.state["app:version"] = "1.0"
    
    # Temp (not persisted, gone after this turn)
    tool_context.state["temp:scratch"] = "intermediate data"
    
    return {"status": "ok"}

# Persistence backend is swappable:
# - InMemorySessionService  (dev)
# - DatabaseSessionService  (production — SQLite, Postgres)
# - VertexAiSessionService  (Google Cloud managed)
# - FirestoreSessionService (Firebase)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — Graph state with checkpointing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from typing import TypedDict, Annotated
from langgraph.graph import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

# State is a typed dictionary — you define the shape
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    raw_input: str
    ticket_draft: dict
    iteration_count: int

# Checkpointing: save full graph state at every node
memory = SqliteSaver.from_conn_string(":memory:")
app = graph.compile(checkpointer=memory)

# "Time-travel debugging" — replay from any checkpoint
# If the app crashes mid-workflow, it resumes from the last checkpoint.
# You can also rewind to any previous state and re-run.

config = {"configurable": {"thread_id": "session-123"}}
result = app.invoke({"messages": ["Create a bug ticket"]}, config)

**Takeaway:** ADK has scoped state (session/user/app/temp) with swappable backends.  
LangGraph has checkpointing and time-travel — superior for long-running, failure-prone workflows.

---
## 4. Tools & Data Integration

**Q: How do you add tools to an agent?**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Tools are plain Python functions
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Just write a function. ADK reads the docstring and type hints
# to generate the tool schema automatically.

async def search_tickets(
    query: str,
    max_results: int,
    tool_context: ToolContext,  # injected automatically
) -> dict:
    """Search JIRA tickets matching the query.
    
    Args:
        query: Search terms.
        max_results: Maximum number of results.
    """
    results = await jira_api.search(query, max_results)
    return {"tickets": results}

# Google-native tools come built-in:
from google.adk.tools import google_search, code_execution
from google.adk.tools import VertexAiSearchTool

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph/LangChain — Tools use the @tool decorator
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from langchain_core.tools import tool

@tool
def search_tickets(query: str, max_results: int) -> dict:
    """Search JIRA tickets matching the query."""
    results = jira_api.search(query, max_results)
    return {"tickets": results}

# LangChain's ecosystem has 700+ pre-built integrations:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import SQLDatabaseToolkit
# ... hundreds more for Slack, Jira, GitHub, databases, etc.

**Takeaway:** ADK is deeply optimised for Google Cloud (Vertex AI, Gemini, BigQuery).  
LangChain/LangGraph inherits a massive community ecosystem of hundreds of integrations.

---
## 5. Agent Collaboration (A2A vs Extensions)

**Q: Can agents from different frameworks talk to each other?**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — A2A protocol (Agent-to-Agent)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ADK supports the A2A protocol natively.
# Any agent can expose an A2A endpoint and be discovered
# by other agents via an Agent Card (like a business card).

# Your ADK agent can delegate to:
# - Another ADK agent on a different server
# - A LangGraph agent that exposes an A2A endpoint  
# - Any framework that implements the A2A spec

# Agent Card example (JSON):
agent_card = {
    "name": "JiraTicketAssistant",
    "description": "Creates JIRA tickets from natural language",
    "url": "https://my-agent.run.app/a2a",
    "capabilities": ["ticket_creation", "ticket_search"],
}

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — No built-in inter-agent protocol
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# LangGraph agents can call each other within the same process
# by nesting one graph inside another:

# Sub-graph as a node
parent_graph.add_node("sub_agent", sub_graph.compile())

# For REMOTE agent communication, you need to build your own
# HTTP layer, or use LangServe / LangGraph Platform.
# There's no standard protocol like A2A — you define the API.

# LangGraph Platform (paid) provides deployment + remote invocation,
# but it's LangChain-ecosystem only, not a cross-framework standard.

**Takeaway:** ADK has A2A as a cross-framework standard. LangGraph keeps it within the LangChain ecosystem.

---
## 6. Observability & Evaluation

**Q: How do you debug and monitor agents?**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Built-in callbacks + Cloud Trace
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Callbacks fire at every lifecycle boundary
agent = LlmAgent(
    name="MyAgent",
    model="gemini-2.5-flash",
    instruction="...",
    before_agent_callback=on_agent_start,   # log agent start
    after_agent_callback=on_agent_end,       # log agent end
    before_tool_callback=on_tool_start,      # log tool calls
    after_tool_callback=on_tool_end,          # log tool results
)

# Callbacks return None to observe, or a value to override.
# ADK also supports OpenTelemetry and Google Cloud Trace natively.

# Built-in evaluation framework:
# - Trajectory evaluation (did the agent call the right tools?)
# - LLM-as-a-Judge (score response quality)
# - AgentEvaluator class for structured test cases

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — LangSmith (separate platform)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os
os.environ["LANGSMITH_API_KEY"] = "your-key"
os.environ["LANGSMITH_TRACING"] = "true"

# Once set, every LLM call, tool call, and graph step is
# automatically traced and sent to the LangSmith dashboard.

# LangSmith provides:
# - Visual trace of every step
# - Latency and cost tracking per node
# - Dataset management for evals
# - Annotation queues for human review
# - Regression testing across prompt versions

# It's purpose-built for agent debugging — arguably the best
# debugging UI in the ecosystem, but it's a separate paid service.

**Takeaway:** ADK fits into existing Google Cloud monitoring (Cloud Trace, OpenTelemetry).  
LangSmith is purpose-built for agent debugging — better UI, but another service to manage.

---
## 7. Security & Compliance

**Q: How does each framework handle auth and security?**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — Inherits Google Cloud security
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# When deployed on Vertex AI / Cloud Run, you get:
# - IAM (Identity & Access Management) for fine-grained permissions
# - IAP (Identity-Aware Proxy) for app-level auth
# - VPC-SC (Service Controls) for data exfiltration prevention
# - Audit logging via Cloud Audit Logs
# - Encryption at rest and in transit by default

# You don't implement any of this — it comes with the platform.
# This is a huge win for enterprise compliance (SOC2, HIPAA, etc.).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — Developer-managed security
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# LangGraph is a library — it doesn't provide security.
# You implement everything yourself:

# - Authentication: add your own middleware (OAuth, JWT, API keys)
# - Authorization: implement role-based access in your code
# - Encryption: configure TLS on your deployment
# - Audit logging: build your own logging pipeline

# This gives total control but requires significant effort.
# For startups: fine. For enterprises with compliance needs: overhead.

**Takeaway:** ADK reduces compliance overhead by inheriting Google Cloud's security stack.  
LangGraph gives total control but you build everything yourself.

---
## 8. Deployment

**Q: How do I deploy my agent to production?**

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ADK — One-command deployment
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Option 1: Vertex AI Agent Engine (fully managed)
# !adk deploy agent_engine --project=my-project --region=us-central1

# Option 2: Cloud Run (containerised)
# !adk deploy cloud_run --project=my-project --region=us-central1

# Option 3: Local dev server
# !adk web
# Opens a browser UI at localhost:8000 with agent selection,
# chat interface, and event viewer.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LangGraph — Self-managed or LangGraph Platform
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Option 1: LangGraph Platform (paid, managed by LangChain)
# - Provides deployment, scaling, monitoring
# - Tight integration with LangSmith

# Option 2: Self-host with FastAPI / LangServe
from langserve import add_routes
from fastapi import FastAPI

fastapi_app = FastAPI()
add_routes(fastapi_app, app)  # app = compiled LangGraph

# Option 3: Deploy on Vertex AI Agent Engine (yes, it supports LangGraph too)
# Both frameworks can run on the same Google infrastructure.

**Takeaway:** Both provide similar architectural options. ADK's CLI (`adk deploy`) is more streamlined for Google Cloud.

---
## 9. Quick Decision Matrix

Use this during Q&A to give quick answers.

In [6]:
%pip install pandas jinja2

  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jinja2]2m1/2 [jinja2]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd

comparison = pd.DataFrame({
    "Question": [
        "We're on Google Cloud",
        "We need complex cyclic workflows",
        "We want crash-resilient long-running agents",
        "Our team prefers OOP / declarative style",
        "We need 100+ third-party integrations",
        "We need cross-framework agent comms",
        "Enterprise compliance is critical",
        "We want the best debugging UI",
        "We're already using LangChain",
        "We want to move fast with common patterns",
    ],
    "Recommendation": [
        "→ ADK",
        "→ LangGraph",
        "→ LangGraph (checkpointing)",
        "→ ADK",
        "→ LangGraph (LangChain ecosystem)",
        "→ ADK (A2A protocol)",
        "→ ADK (Google Cloud IAM/VPC-SC)",
        "→ LangGraph (LangSmith)",
        "→ LangGraph (natural extension)",
        "→ ADK (opinionated primitives)",
    ],
})

# Style the dataframe for presentation
comparison.style.set_properties(**{
    'text-align': 'left',
    'font-size': '13px',
}).hide(axis='index')

Question,Recommendation
We're on Google Cloud,→ ADK
We need complex cyclic workflows,→ LangGraph
We want crash-resilient long-running agents,→ LangGraph (checkpointing)
Our team prefers OOP / declarative style,→ ADK
We need 100+ third-party integrations,→ LangGraph (LangChain ecosystem)
We need cross-framework agent comms,→ ADK (A2A protocol)
Enterprise compliance is critical,→ ADK (Google Cloud IAM/VPC-SC)
We want the best debugging UI,→ LangGraph (LangSmith)
We're already using LangChain,→ LangGraph (natural extension)
We want to move fast with common patterns,→ ADK (opinionated primitives)


---

## Summary

| | **ADK** | **LangGraph** |
|---|---|---|
| **Philosophy** | Object-oriented, declarative | Graph-based, functional |
| **Best for** | GCP teams, common patterns, enterprise compliance | Complex workflows, fault tolerance, LangChain ecosystem |
| **Learning curve** | Easier (OOP devs) | Steeper (graph thinking) |
| **Vendor tie-in** | Google Cloud optimised | Cloud-agnostic |

**Both are excellent.** The choice is about fit, not quality.

---
*For internal use only.*